# Sanity check: does ALFRED vintage BAA - GS10 match FRED's BAA10Y series?

`BAA10Y` ("Moody's Seasoned Baa Corporate Bond Yield Relative to Yield on 10-Year Treasury Constant Maturity") is **daily**, while FRED's `BAA` and `GS10` series are **monthly** -- so `BAA10Y` is not literally `BAA - GS10` on matching frequency.

This notebook pulls the **ALFRED first-release (as-originally-published) vintage** of monthly `BAA` and `GS10` (i.e. the value as it stood right after each observation was first published, before any later revisions), computes `BAA - GS10`, and compares it against `BAA10Y` resampled to a monthly mean -- to see how close the monthly credit spread implied by the vintage `BAA`/`GS10` data is to `BAA10Y`, and to make the vintage nature of the ALFRED pull visible in a table.

In [16]:
import os

import pandas as pd
import requests
import truststore

truststore.inject_into_ssl()

pd.set_option("display.max_rows", 50)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

In [17]:
FRED_OBSERVATIONS_URL = "https://api.stlouisfed.org/fred/series/observations"

START = "2000-01-01"
END = "2026-07-31"
TOLERANCE = 0.02  # pp; monthly BAA/GS10 vs monthly-mean-of-daily BAA10Y are not identical constructs

## Helpers: API key + FRED/ALFRED fetchers

In [18]:
def load_dotenv_fallback(path=None):
    """Minimal .env loader so FRED_API_KEY doesn't need to be exported manually."""
    if path is None:
        repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
        path = os.path.join(repo_root, ".env")
    if not os.path.exists(path):
        return
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, _, value = line.partition("=")
            os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


def get_fred_api_key():
    load_dotenv_fallback()
    api_key = os.getenv("FRED_API_KEY")
    if not api_key:
        raise RuntimeError(
            "Missing FRED_API_KEY. Set it in your environment (.env) before running this notebook."
        )
    return api_key

In [19]:
def fetch_alfred_series(series_id, api_key):
    """Fetch the ALFRED first-release (as-originally-published) vintage of a series.

    output_type=4 returns, for each observation date, only its initial
    release value -- the standard "vintage" read for a series.
    """
    params = {
        "api_key": api_key,
        "file_type": "json",
        "series_id": series_id,
        "output_type": 4,
        "realtime_start": START,
        "realtime_end": END,
        "observation_start": START,
        "observation_end": END,
        "sort_order": "asc",
    }
    response = requests.get(FRED_OBSERVATIONS_URL, params=params, timeout=30)
    response.raise_for_status()
    payload = response.json()
    if "error_message" in payload:
        raise RuntimeError(f"FRED API error for {series_id}: {payload['error_message']}")

    df = pd.DataFrame(payload["observations"])
    df["date"] = pd.to_datetime(df["date"])
    df["value"] = pd.to_numeric(df["value"].replace(".", pd.NA), errors="coerce")
    return df.set_index("date")["value"].sort_index()


def fetch_alfred_series_full(series_id, api_key):
    """Fetch ALL vintages (every realtime_start/realtime_end revision) of a series,
    so we can show, observation by observation, how the published value changed
    over time -- i.e. that ALFRED is genuinely vintage/real-time data, not a
    single static series.
    """
    params = {
        "api_key": api_key,
        "file_type": "json",
        "series_id": series_id,
        "output_type": 2,  # all vintages, one row per (date, realtime_start)
        "realtime_start": START,
        "realtime_end": END,
        "observation_start": START,
        "observation_end": END,
        "sort_order": "asc",
    }
    response = requests.get(FRED_OBSERVATIONS_URL, params=params, timeout=30)
    response.raise_for_status()
    payload = response.json()
    if "error_message" in payload:
        raise RuntimeError(f"FRED API error for {series_id}: {payload['error_message']}")

    df = pd.DataFrame(payload["observations"])
    df["date"] = pd.to_datetime(df["date"])
    df["realtime_start"] = pd.to_datetime(df["realtime_start"])
    df["value"] = pd.to_numeric(df["value"].replace(".", pd.NA), errors="coerce")
    return df


def fetch_fred_series(series_id, api_key):
    params = {
        "api_key": api_key,
        "file_type": "json",
        "series_id": series_id,
        "observation_start": START,
        "observation_end": END,
        "sort_order": "asc",
    }
    response = requests.get(FRED_OBSERVATIONS_URL, params=params, timeout=30)
    response.raise_for_status()
    payload = response.json()
    if "error_message" in payload:
        raise RuntimeError(f"FRED API error for {series_id}: {payload['error_message']}")

    df = pd.DataFrame(payload["observations"])
    df["date"] = pd.to_datetime(df["date"])
    df["value"] = pd.to_numeric(df["value"].replace(".", pd.NA), errors="coerce")
    return df.set_index("date")["value"].sort_index()

## Fetch data

In [20]:
api_key = get_fred_api_key()

print("Fetching ALFRED vintage BAA (first release) ...")
baa = fetch_alfred_series("BAA", api_key)

print("Fetching ALFRED vintage GS10 (first release) ...")
gs10 = fetch_alfred_series("GS10", api_key)

print("Fetching FRED BAA10Y (daily) ...")
baa10y_daily = fetch_fred_series("BAA10Y", api_key)

Fetching ALFRED vintage BAA (first release) ...
Fetching ALFRED vintage GS10 (first release) ...
Fetching FRED BAA10Y (daily) ...


## Compute monthly BAA - GS10 (vintage) vs published BAA10Y

In [21]:
computed = (baa - gs10).dropna()
computed.index = computed.index.to_period("M").to_timestamp("M")

published = baa10y_daily.dropna().resample("ME").mean()
published.index.name = "date"

compare = pd.DataFrame(
    {"computed_BAA_minus_GS10": computed, "published_BAA10Y_monthly_mean": published}
).dropna()
compare["diff"] = (
    compare["computed_BAA_minus_GS10"] - compare["published_BAA10Y_monthly_mean"]
)
compare["abs_diff"] = compare["diff"].abs()
compare["match"] = compare["abs_diff"] < TOLERANCE

print(f"Compared {len(compare)} overlapping months")
print(f"Max abs diff:  {compare['abs_diff'].max():.4f}")
print(f"Mean abs diff: {compare['abs_diff'].mean():.4f}")
print(f"Correlation:   {compare['computed_BAA_minus_GS10'].corr(compare['published_BAA10Y_monthly_mean']):.6f}")
print(f"Matching (|diff| < {TOLERANCE}pp): {compare['match'].sum()} / {len(compare)}")

Compared 318 overlapping months
Max abs diff:  0.0264
Mean abs diff: 0.0033
Correlation:   0.999983
Matching (|diff| < 0.02pp): 317 / 318


### Full comparison table

### Largest mismatches (if any)

In [22]:
compare.reindex(compare["abs_diff"].sort_values(ascending=False).index).head(25)

,computed_BAA_minus_GS10,published_BAA10Y_monthly_mean,diff,abs_diff,match
date,,,,,
2008-12-31,6.0400,6.0136,0.0264,0.0264,False
2021-04-30,1.9600,1.9724,-0.0124,0.0124,True
2016-06-30,2.8900,2.8809,0.0091,0.0091,True
2015-04-30,2.5400,2.5491,-0.0091,0.0091,True
2014-04-30,2.1900,2.1990,-0.0090,0.0090,True
2004-02-29,2.1900,2.1811,0.0089,0.0089,True
2001-03-31,2.9500,2.9586,-0.0086,0.0086,True
2024-10-31,1.5300,1.5386,-0.0086,0.0086,True
2025-08-31,1.7400,1.7314,0.0086,0.0086,True
